In [58]:
import joblib

joblib.dump(
    model_pipeline_v2,
    "models/construction_risk_prediction_model.pkl"
)

print("Model saved successfully")

Model saved successfully


In [57]:
import os

os.makedirs("models", exist_ok=True)

print("Models folder ready")

Models folder ready


In [56]:
risk_results["Risk_Level"].value_counts()

Risk_Level
Low Risk       2117
High Risk       222
Medium Risk     146
Name: count, dtype: int64

In [55]:
risk_results = X_test_v2.copy()

risk_results["Actual_Overdue"] = y_test_v2.values

risk_results["Risk_Probability"] = y_prob_v2

risk_results["Risk_Level"] = pd.cut(
    risk_results["Risk_Probability"],
    bins=[0,0.3,0.7,1],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

risk_results.head()

,Type,To Package,Priority,Cause,Task Group,Days_Open,Days_Since_Status_Change,Project_Total_Forms,Project_Total_Actions,Project_Open_Actions,Project_Overdue_Forms,Has_Image,Has_Comment,Has_Document,Actual_Overdue,Risk_Probability,Risk_Level
11125,JPC - Progress Photo,Piling,NaN,NaN,Site Management,77,77,744,938,20,0,1,1,1,0,0.000739,Low Risk
10507,Safety Notice (Amber) - General Issue,Main Contractor,NaN,JPC - Safety - House Keeping,Safety,4,4,744,938,20,0,1,1,1,0,0.000119,Low Risk
12318,Safety Notice (Amber) - General Issue,Formwork,NaN,JPC - Quality - Workmanship,Safety,55,55,396,357,8,0,1,0,0,0,0.000512,Low Risk
2556,Safety Notice (Green) - Good Observation,Scaffolding,NaN,NaN,Safety,320,320,4043,2945,136,0,1,1,1,0,0.000621,Low Risk
2946,Safety Notice (Amber) - General Issue,Main Contractor,System Failure,JPC - Safety - House Keeping,Safety,363,361,4043,2945,136,0,1,1,1,0,0.000457,Low Risk


In [54]:
from sklearn.inspection import permutation_importance

importance_v2 = permutation_importance(
    model_pipeline_v2,
    X_test_v2,
    y_test_v2,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc"
)

importance_df = pd.DataFrame({
    "Feature": X_test_v2.columns,
    "Importance": importance_v2.importances_mean
})

importance_df.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
4,Task Group,0.218709
5,Days_Open,0.038484
0,Type,0.028824
6,Days_Since_Status_Change,0.015375
9,Project_Open_Actions,0.009914
1,To Package,0.002631
3,Cause,0.001105
2,Priority,0.000326
7,Project_Total_Forms,0.000305
12,Has_Comment,0.000273


In [53]:
y_pred_v2 = model_pipeline_v2.predict(
    X_test_v2
)

y_prob_v2 = model_pipeline_v2.predict_proba(
    X_test_v2
)[:,1]


print(classification_report(
    y_test_v2,
    y_pred_v2
))


print(
    "ROC-AUC:",
    roc_auc_score(
        y_test_v2,
        y_prob_v2
    )
)

              precision    recall  f1-score   support

           0       1.00      0.95      0.97      2313
           1       0.59      0.96      0.73       172

    accuracy                           0.95      2485
   macro avg       0.79      0.95      0.85      2485
weighted avg       0.97      0.95      0.96      2485

ROC-AUC: 0.9914198312872642


In [52]:
model_pipeline_v2.fit(
    X_train_v2,
    y_train_v2
)

print("Early warning model trained successfully")

Early warning model trained successfully


In [51]:
model_pipeline_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", xgb_model_v2)
    ]
)

In [50]:
xgb_model_v2 = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    scale_pos_weight=13.48,
    eval_metric="logloss"
)

In [49]:
preprocessor_v2 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features_v2
        )
    ],
    remainder="passthrough"
)

In [48]:
categorical_features_v2 = X_train_v2.select_dtypes(
    include=["object"]
).columns.tolist()


numeric_features_v2 = X_train_v2.select_dtypes(
    exclude=["object"]
).columns.tolist()


print(categorical_features_v2)
print(numeric_features_v2)

['Type', 'To Package', 'Priority', 'Cause', 'Task Group']
['Days_Open', 'Days_Since_Status_Change', 'Project_Total_Forms', 'Project_Total_Actions', 'Project_Open_Actions', 'Project_Overdue_Forms', 'Has_Image', 'Has_Comment', 'Has_Document']


In [47]:
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train_v2.shape)
print(X_test_v2.shape)

(9939, 14)
(2485, 14)


In [46]:
X_v2 = X.drop(
    columns=[
        "Status",
        "Report Status"
    ]
)

X_v2.shape

(12424, 14)

In [45]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model_pipeline,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc"
)

importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": result.importances_mean
})

importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
6,Task Group,5.432565e-02
0,Status,1.452910e-02
5,Report Status,1.216934e-02
7,Days_Open,1.685871e-03
1,Type,9.612001e-04
2,To Package,1.146201e-04
4,Cause,7.993243e-05
11,Project_Open_Actions,5.680733e-05
9,Project_Total_Forms,2.010879e-05
8,Days_Since_Status_Change,1.407615e-05


In [44]:
print(classification_report(
    y_test,
    y_pred
))

print(
    "ROC-AUC:",
    roc_auc_score(y_test, y_prob)
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2313
           1       0.99      1.00      1.00       172

    accuracy                           1.00      2485
   macro avg       1.00      1.00      1.00      2485
weighted avg       1.00      1.00      1.00      2485

ROC-AUC: 0.9999296192400888


In [43]:
y_pred = model_pipeline.predict(X_test)

y_prob = model_pipeline.predict_proba(
    X_test
)[:,1]

print("Predictions generated")

Predictions generated


In [42]:
model_pipeline.fit(
    X_train,
    y_train
)

print("Model trained successfully")

Model trained successfully


In [41]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

In [40]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    scale_pos_weight=13.48,
    eval_metric="logloss"
)

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [38]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()


numeric_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()


print("Categorical:")
print(categorical_features)

print("\nNumeric:")
print(numeric_features)

Categorical:
['Status', 'Type', 'To Package', 'Priority', 'Cause', 'Report Status', 'Task Group']

Numeric:
['Days_Open', 'Days_Since_Status_Change', 'Project_Total_Forms', 'Project_Total_Actions', 'Project_Open_Actions', 'Project_Overdue_Forms', 'Has_Image', 'Has_Comment', 'Has_Document']


In [37]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining overdue rate:")
print(y_train.mean())

print("\nTesting overdue rate:")
print(y_test.mean())

Training set: (9939, 16)
Testing set: (2485, 16)

Training overdue rate:
0.06902102827246202

Testing overdue rate:
0.06921529175050302


In [35]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining overdue rate:")
print(y_train.mean())

print("\nTesting overdue rate:")
print(y_test.mean())

NameError: name 'X_train' is not defined

In [34]:
X = X.drop(
    columns=[
        "Images",
        "Comments",
        "Documents",
        "project"
    ]
)

X.shape

(12424, 16)

In [33]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


numeric_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()


print("Categorical:")
print(categorical_features)

print("\nNumeric:")
print(numeric_features)

Categorical:
['Status', 'Type', 'To Package', 'Images', 'Comments', 'Documents', 'Priority', 'Cause', 'Report Status', 'Task Group']

Numeric:
['project', 'Days_Open', 'Days_Since_Status_Change', 'Project_Total_Forms', 'Project_Total_Actions', 'Project_Open_Actions', 'Project_Overdue_Forms', 'Has_Image', 'Has_Comment', 'Has_Document']


In [32]:
X = X.drop(
    columns=[
        "Created",
        "Status Changed"
    ]
)

X.shape

(12424, 20)

In [31]:
X.dtypes

Status                              object
Created                     datetime64[ns]
Type                                object
To Package                          object
Status Changed              datetime64[ns]
Images                              object
Comments                            object
Documents                           object
Priority                            object
Cause                               object
project                              int64
Report Status                       object
Task Group                          object
Days_Open                            int64
Days_Since_Status_Change             int64
Project_Total_Forms                  int64
Project_Total_Actions                int64
Project_Open_Actions                 int64
Project_Overdue_Forms                int64
Has_Image                            int64
Has_Comment                          int64
Has_Document                         int64
dtype: object

In [30]:
model_data = tasks_clean.copy()


features_to_drop = [
    "Ref",
    "Location",
    "Description",
    "Target",
    "Association",
    "OverDue",
    "Project"
]


X = model_data.drop(
    columns=features_to_drop
)

y = model_data["OverDue"].astype(int)


print("Features:", X.shape)
print("Target:", y.shape)

Features: (12424, 22)
Target: (12424,)


In [29]:
overdue_count = tasks_clean["OverDue"].value_counts()

scale_weight = overdue_count[False] / overdue_count[True]

scale_weight

np.float64(13.48018648018648)

In [28]:
tasks_clean["OverDue"].value_counts(normalize=True)

OverDue
False    0.93094
True     0.06906
Name: proportion, dtype: float64

In [27]:
tasks_clean["OverDue"].value_counts()

OverDue
False    11566
True       858
Name: count, dtype: int64

In [26]:
tasks_clean["Has_Image"] = (
    tasks_clean["Images"].notna()
).astype(int)

tasks_clean["Has_Comment"] = (
    tasks_clean["Comments"].notna()
).astype(int)

tasks_clean["Has_Document"] = (
    tasks_clean["Documents"].notna()
).astype(int)


tasks_clean[
[
"Has_Image",
"Has_Comment",
"Has_Document"
]
].head()

,Has_Image,Has_Comment,Has_Document
0,0,0,0
1,1,1,1
2,1,1,1
3,1,1,1
4,1,1,1


In [25]:
tasks_clean = tasks_clean.drop(
    columns=["Project_Overdue_Rate"]
)

tasks_clean.shape

(12424, 26)

In [24]:
tasks_clean[
[
"Project_Total_Forms",
"Project_Total_Actions",
"Project_Open_Actions",
"Project_Overdue_Rate"
]
].describe()

,Project_Total_Forms,Project_Total_Actions,Project_Open_Actions,Project_Overdue_Rate
count,12424.000000,12424.000000,12424.000000,12424.0
mean,2129.765695,1841.918303,89.336929,0.0
std,1405.260123,981.919486,55.968818,0.0
min,396.000000,357.000000,8.000000,0.0
25%,744.000000,938.000000,48.000000,0.0
50%,2149.000000,2262.000000,114.000000,0.0
75%,4043.000000,2945.000000,136.000000,0.0
max,4043.000000,2945.000000,181.000000,0.0


In [23]:
tasks_clean = tasks_clean.merge(
    project_risk,
    left_on="project",
    right_on="Project",
    how="left"
)

tasks_clean.shape

(12424, 27)

In [22]:
project_risk = forms_clean.groupby("Project").agg(
    Project_Total_Forms=("Ref", "count"),
    Project_Total_Actions=("Total Actions", "sum"),
    Project_Open_Actions=("Open Actions", "sum"),
    Project_Overdue_Forms=("OverDue", "sum")
).reset_index()


project_risk["Project_Overdue_Rate"] = (
    project_risk["Project_Overdue_Forms"] /
    project_risk["Project_Total_Forms"]
)


project_risk.head()

,Project,Project_Total_Forms,Project_Total_Actions,Project_Open_Actions,Project_Overdue_Forms,Project_Overdue_Rate
0,1328,4043,2945,136,0,0.0
1,1329,1212,422,30,0,0.0
2,1330,2149,2262,48,0,0.0
3,1335,804,945,114,0,0.0
4,1338,510,639,181,0,0.0


In [21]:
today = tasks_clean["Status Changed"].max()

tasks_clean["Days_Open"] = (
    today - tasks_clean["Created"]
).dt.days


tasks_clean["Days_Since_Status_Change"] = (
    today - tasks_clean["Status Changed"]
).dt.days


tasks_clean[[
    "Days_Open",
    "Days_Since_Status_Change"
]].describe()

,Days_Open,Days_Since_Status_Change
count,12424.000000,12424.000000
mean,146.394317,135.338619
std,123.919569,120.823621
min,0.000000,0.000000
25%,48.000000,39.000000
50%,102.000000,90.000000
75%,222.000000,204.000000
max,543.000000,543.000000


In [20]:
tasks_clean["Created"] = pd.to_datetime(
    tasks_clean["Created"],
    dayfirst=True
)

tasks_clean["Status Changed"] = pd.to_datetime(
    tasks_clean["Status Changed"],
    dayfirst=True
)


forms_clean["Created"] = pd.to_datetime(
    forms_clean["Created"],
    dayfirst=True
)

forms_clean["Status Changed"] = pd.to_datetime(
    forms_clean["Status Changed"],
    dayfirst=True
)


print("Dates converted successfully")

Dates converted successfully


In [19]:
tasks_clean = tasks.copy()
forms_clean = forms.copy()

print(tasks_clean.shape)
print(forms_clean.shape)

(12424, 19)
(10254, 17)


In [18]:
forms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10254 entries, 0 to 10253
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Ref                  10254 non-null  object
 1   Status               10254 non-null  object
 2   Location             10254 non-null  object
 3   Name                 10254 non-null  object
 4   Created              10254 non-null  object
 5   Type                 10254 non-null  object
 6   Status Changed       10254 non-null  object
 7   Open Actions         10254 non-null  int64 
 8   Total Actions        10254 non-null  int64 
 9   Association          2098 non-null   object
 10  OverDue              10254 non-null  bool  
 11  Images               10254 non-null  bool  
 12  Comments             10254 non-null  bool  
 13  Documents            9450 non-null   object
 14  Project              10254 non-null  int64 
 15  Report Forms Status  10252 non-null  object
 16  Repo

In [17]:
tasks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12424 entries, 0 to 12423
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Ref             12424 non-null  object 
 1   Status          12424 non-null  object 
 2   Location        12424 non-null  object 
 3   Description     12424 non-null  object 
 4   Created         12424 non-null  object 
 5   Target          2568 non-null   float64
 6   Type            12424 non-null  object 
 7   To Package      11382 non-null  object 
 8   Status Changed  12424 non-null  object 
 9   Association     9483 non-null   object 
 10  OverDue         12424 non-null  bool   
 11  Images          12272 non-null  object 
 12  Comments        11902 non-null  object 
 13  Documents       11780 non-null  object 
 14  Priority        2366 non-null   object 
 15  Cause           9683 non-null   object 
 16  project         12424 non-null  int64  
 17  Report Status   12424 non-null 

In [16]:
print("Forms missing values")
display(forms.isnull().sum())

print("Tasks missing values")
display(tasks.isnull().sum())

Forms missing values


Ref                       0
Status                    0
Location                  0
Name                      0
Created                   0
Type                      0
Status Changed            0
Open Actions              0
Total Actions             0
Association            8156
OverDue                   0
Images                    0
Comments                  0
Documents               804
Project                   0
Report Forms Status       2
Report Forms Group        4
dtype: int64

Tasks missing values


Ref                   0
Status                0
Location              0
Description           0
Created               0
Target             9856
Type                  0
To Package         1042
Status Changed        0
Association        2941
OverDue               0
Images              152
Comments            522
Documents           644
Priority          10058
Cause              2741
project               0
Report Status         0
Task Group           50
dtype: int64

In [15]:
print("FORMS COLUMNS")
print(forms.columns.tolist())

print("\nTASKS COLUMNS")
print(tasks.columns.tolist())

FORMS COLUMNS
['Ref', 'Status', 'Location', 'Name', 'Created', 'Type', 'Status Changed', 'Open Actions', 'Total Actions', 'Association', 'OverDue', 'Images', 'Comments', 'Documents', 'Project', 'Report Forms Status', 'Report Forms Group']

TASKS COLUMNS
['Ref', 'Status', 'Location', 'Description', 'Created', 'Target', 'Type', 'To Package', 'Status Changed', 'Association', 'OverDue', 'Images', 'Comments', 'Documents', 'Priority', 'Cause', 'project', 'Report Status', 'Task Group']


In [14]:
tasks.head()

,Ref,Status,Location,Description,Created,Target,Type,To Package,Status Changed,Association,OverDue,Images,Comments,Documents,Priority,Cause,project,Report Status,Task Group
0,T1.23963030,Open,JPC Project Management>EHS Management>01 Inspe...,task raised in incorrect location of this form...,14/09/2020,NaN,Safety Notice (Amber) - General Issue,Main Contractor,14/09/2020,FormAnswer,False,NaN,NaN,NaN,Behavioural Failure,JPC - Safety - Documentation,1328,Open,Safety
1,T116412.200,Closed,QC & BC(A)R>ITP 02 Architectural & M&E Service...,Metsec,14/09/2020,NaN,JPC - Progress Photo,Ceilings & Partitions,14/09/2020,NaN,False,True,False,False,NaN,NaN,1328,Closed,Site Management
2,T141663.27,EHS Good Observation,JPC Project Management>EHS Management>01 Inspe...,Good clear exclusion zones and access through ...,14/09/2020,NaN,Safety Notice (Green) - Good Observation,Main Contractor,14/09/2020,FormAnswer,False,True,False,False,NaN,JPC - Safety - Access,1328,Closed,Safety
3,T116412.199,Closed,QC & BC(A)R>ITP 02 Architectural & M&E Service...,RC walls,14/09/2020,NaN,JPC - Progress Photo,Precast Concrete,14/09/2020,NaN,False,True,False,False,NaN,NaN,1328,Closed,Site Management
4,T141663.26,EHS Good Observation,JPC Project Management>EHS Management>01 Inspe...,"block 02 working level has good housekeeping, ...",14/09/2020,NaN,Safety Notice (Green) - Good Observation,Precast Concrete,14/09/2020,FormAnswer,False,True,False,False,NaN,JPC - Safety - House Keeping,1328,Closed,Safety


In [13]:
forms.head()

,Ref,Status,Location,Name,Created,Type,Status Changed,Open Actions,Total Actions,Association,OverDue,Images,Comments,Documents,Project,Report Forms Status,Report Forms Group
0,F145185.4,Opened,01 Daily Site Diary>Site Management>JPC Projec...,1328 CM-SM-FRM-001 Site Diary,15/09/2020,Site Management,15/09/2020,0,0,NaN,False,True,False,False,1328,Open,Site Management
1,F1.495500,Open / Ongoing Works,02 Daily Work Plan>Site Management>JPC Project...,SM-FRM-SUB-101 Daily Work Plan,15/09/2020,Subcontractor Inspections,15/09/2020,0,0,NaN,False,False,False,False,1328,Open,Subcontractor
2,F1.495499,Open / Ongoing Works,02 Daily Work Plan>Site Management>JPC Project...,SM-FRM-SUB-101 Daily Work Plan,15/09/2020,Subcontractor Inspections,15/09/2020,0,0,NaN,False,False,False,False,1328,Open,Subcontractor
3,F1.495498,Open / Ongoing Works,02 Daily Work Plan>Site Management>JPC Project...,SM-FRM-SUB-101 Daily Work Plan,15/09/2020,Subcontractor Inspections,15/09/2020,0,0,NaN,False,False,False,False,1328,Open,Subcontractor
4,F1.495496,Open / Ongoing Works,02 Daily Work Plan>Site Management>JPC Project...,SM-FRM-SUB-101 Daily Work Plan,15/09/2020,Subcontractor Inspections,15/09/2020,0,0,NaN,False,False,False,False,1328,Open,Subcontractor


In [12]:
forms = pd.read_csv(
    "../data/raw/Construction_Data_PM_Forms_All_Projects.csv"
)

tasks = pd.read_csv(
    "../data/raw/Construction_Data_PM_Tasks_All_Projects.csv"
)

print("Forms shape:", forms.shape)
print("Tasks shape:", tasks.shape)

Forms shape: (10254, 17)
Tasks shape: (12424, 19)


In [7]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

print("Libraries loaded successfully")

Libraries loaded successfully
